# Agent-Based Muay Thai Multiple Combos Classifier

This notebook evaluates the LangGraph agent approach (from `multiple_agent/basic_agent_explicit.py`) to serve as a comparison for the DSPy and baseline classifiers on videos with multiple combos. It uses the same data splits and evaluation metrics.


## Section 1: Setup and Configuration


In [1]:
import json
from pathlib import Path
import os
from dotenv import load_dotenv
import random
import time
from typing import List, Dict, Tuple
import re
from collections import defaultdict

# Import agent from basic_agent_explicit
import sys
sys.path.append('multiple_agent')  # Add agent directory to path
from basic_agent_explicit import graph, llm
from langchain_core.messages import HumanMessage

# Load environment variables
load_dotenv()

# Data paths (same as other notebooks)
DATA_ROOT = Path("data/multiple_combos")
SHADOW_DIR = DATA_ROOT / "shadow"
HEAVY_BAG_DIR = DATA_ROOT / "heavy_bag"

# Training split sizes (3 shadow + 4 heavy_bag = 7 total for training)
SHADOW_TRAIN_SIZE = 3
HEAVY_BAG_TRAIN_SIZE = 4

# Rate limiting for free tier
# Agent makes multiple API calls (analyze_video + potentially extract_pose_data)
# Use 60 seconds between videos to avoid rate limits
REQUEST_DELAY = 60

print("Setup complete!")
print(f"Data directories: {SHADOW_DIR}, {HEAVY_BAG_DIR}")
print(f"Rate limiting: {REQUEST_DELAY}s delay between videos")
print(f"Agent graph loaded successfully")


Setup complete!
Data directories: data\multiple_combos\shadow, data\multiple_combos\heavy_bag
Rate limiting: 60s delay between videos
Agent graph loaded successfully


## Section 2: Data Loading


In [2]:
def load_labels(json_path: Path) -> Dict[str, List[List[str]]]:
    """Load true labels from JSON file. For multiple_combos, each video has multiple combos."""
    with open(json_path, 'r') as f:
        return json.load(f)

def create_examples(labels: Dict[str, List[List[str]]], video_dir: Path, dataset_name: str) -> List[Dict]:
    """Create example dictionaries from labels and video directory.
    
    For multiple_combos, each video has multiple combos (list of lists).
    We keep the combos structure as arrays of arrays for evaluation.
    """
    examples = []
    for video_id, combos in labels.items():
        video_path = video_dir / f"{video_id}.mov"
        if video_path.exists():
            example = {
                'video_path': str(video_path),
                'combos': combos  # Keep as array of arrays
            }
            examples.append(example)
        else:
            print(f"Warning: Video {video_path} not found")
    return examples

# Load labels from both datasets
shadow_labels = load_labels(SHADOW_DIR / "true_labels.json")
heavy_bag_labels = load_labels(HEAVY_BAG_DIR / "true_labels.json")

# Create examples
shadow_examples = create_examples(shadow_labels, SHADOW_DIR, "shadow")
heavy_bag_examples = create_examples(heavy_bag_labels, HEAVY_BAG_DIR, "heavy_bag")

print(f"Loaded {len(shadow_examples)} shadow examples")
print(f"Loaded {len(heavy_bag_examples)} heavy_bag examples")


Loaded 12 shadow examples
Loaded 19 heavy_bag examples


In [3]:
# Split data into train and validation
# Use the same random seed (42) as DSPy and baseline notebooks for reproducibility
# Training set: 3 shadow + 4 heavy_bag = 7 total

random.seed(42)  # Same seed as other notebooks

# Shadow split (3 for training)
shadow_indices = list(range(len(shadow_examples)))
random.shuffle(shadow_indices)
shadow_train_indices = set(shadow_indices[:SHADOW_TRAIN_SIZE])
shadow_train = [shadow_examples[i] for i in shadow_train_indices]
shadow_val = [shadow_examples[i] for i in range(len(shadow_examples)) if i not in shadow_train_indices]

# Heavy bag split (4 for training)
heavy_bag_indices = list(range(len(heavy_bag_examples)))
random.shuffle(heavy_bag_indices)
heavy_bag_train_indices = set(heavy_bag_indices[:HEAVY_BAG_TRAIN_SIZE])
heavy_bag_train = [heavy_bag_examples[i] for i in heavy_bag_train_indices]
heavy_bag_val = [heavy_bag_examples[i] for i in range(len(heavy_bag_examples)) if i not in heavy_bag_train_indices]

# Combine train and validation sets
train_examples = shadow_train + heavy_bag_train
val_examples = shadow_val + heavy_bag_val

print(f"Training set: {len(train_examples)} examples ({len(shadow_train)} shadow + {len(heavy_bag_train)} heavy_bag)")
print(f"Validation set: {len(val_examples)} examples ({len(shadow_val)} shadow + {len(heavy_bag_val)} heavy_bag)")

# Display some statistics
all_strikes = set()
total_combos = 0
for ex in train_examples + val_examples:
    for combo in ex['combos']:
        all_strikes.update(combo)
        total_combos += 1

print(f"\nUnique strike types: {sorted(all_strikes)}")
print(f"Total unique strike types: {len(all_strikes)}")
print(f"Total combos across all videos: {total_combos}")
print(f"Average combos per video: {total_combos / len(train_examples + val_examples):.2f}")


Training set: 7 examples (3 shadow + 4 heavy_bag)
Validation set: 24 examples (9 shadow + 15 heavy_bag)

Unique strike types: ['cross', 'elbow', 'jab', 'knee', 'lead_elbow', 'lead_hook', 'lead_roundhouse', 'lead_uppercut', 'low_kick', 'rear_elbow', 'rear_hook', 'rear_uppercut', 'roundhouse', 'spinning_elbow', 'switch_kick', 'switch_knee', 'teep']
Total unique strike types: 17
Total combos across all videos: 93
Average combos per video: 3.00


## Section 3: Agent Classification Function


In [4]:
def classify_video_agent(video_path: str) -> List[List[str]]:
    """
    Classify Muay Thai strikes in a video using the LangGraph agent.
    Returns a list of combos, where each combo is a list of strike names.
    This tests the agent's ability to recognize when one combo ends and another begins.
    """
    try:
        # Invoke the agent
        result = graph.invoke(
            {"messages": [HumanMessage(content=f"Classify the strikes in this video: {video_path}")]}
        )
        
        # Extract response from the last message
        agent_response = result["messages"][-1].content
        
        # Handle different response formats
        # Response might be a string, list, or dict
        if isinstance(agent_response, str):
            response_text = agent_response
        elif isinstance(agent_response, list):
            # If response is a list of content blocks, extract text
            response_text = ""
            for item in agent_response:
                if isinstance(item, dict) and 'text' in item:
                    response_text += item['text']
                elif isinstance(item, str):
                    response_text += item
        elif isinstance(agent_response, dict):
            # If response is a dict, try to extract text
            response_text = agent_response.get('text', str(agent_response))
        else:
            response_text = str(agent_response)
        
        # Try to extract JSON array of arrays from response
        # First, try direct JSON parsing
        try:
            combos = json.loads(response_text.strip())
            if isinstance(combos, list):
                # Validate that it's an array of arrays
                if all(isinstance(combo, list) for combo in combos):
                    return combos
                # If it's a flat array, wrap it in a single combo
                elif all(isinstance(strike, str) for strike in combos):
                    return [combos]
        except (json.JSONDecodeError, ValueError):
            pass
        
        # If direct parsing fails, try to find JSON array of arrays in the text
        # Look for nested arrays: [[...], [...]]
        json_match = re.search(r'\[(\[.*?\](?:,\s*\[.*?\])*)\]', response_text, re.DOTALL)
        if json_match:
            try:
                combos = json.loads(json_match.group())
                if isinstance(combos, list) and all(isinstance(combo, list) for combo in combos):
                    return combos
            except json.JSONDecodeError:
                pass
        
        # Try to find a single array and wrap it
        json_match = re.search(r'\[.*?\]', response_text)
        if json_match:
            try:
                strikes = json.loads(json_match.group())
                if isinstance(strikes, list) and all(isinstance(strike, str) for strike in strikes):
                    # Treat as a single combo
                    return [strikes]
            except json.JSONDecodeError:
                pass
        
        # If still no luck, return empty list
        print(f"Warning: Could not parse JSON from agent response: {response_text[:200]}")
        return []
        
    except Exception as e:
        print(f"Error classifying video {video_path}: {e}")
        return []

print("Agent classification function created!")


Agent classification function created!


## Section 4: Evaluation


In [5]:
# Evaluation metrics for arrays of arrays (multiple combos)

def exact_match_metric(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> bool:
    """Evaluate exact match between predicted and true combos (arrays of arrays)."""
    return true_combos == predicted_combos

def combo_count_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate accuracy of combo count (how many combos detected vs actual)."""
    if not true_combos:
        return 1.0 if not predicted_combos else 0.0
    if not predicted_combos:
        return 0.0
    
    # Penalize based on difference in count
    count_diff = abs(len(true_combos) - len(predicted_combos))
    max_count = max(len(true_combos), len(predicted_combos))
    return 1.0 - (count_diff / max_count) if max_count > 0 else 1.0

def combo_boundary_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate accuracy of combo boundaries (whether combos are correctly separated).
    
    This flattens both and checks if the model correctly identified where combos start/end.
    """
    # Flatten true combos with markers
    true_flat = []
    true_boundaries = set()
    pos = 0
    for combo in true_combos:
        if combo:  # Only add boundary if combo is non-empty
            true_boundaries.add(pos)
            true_flat.extend(combo)
            pos += len(combo)
    
    # Flatten predicted combos with markers
    pred_flat = []
    pred_boundaries = set()
    pos = 0
    for combo in predicted_combos:
        if combo:  # Only add boundary if combo is non-empty
            pred_boundaries.add(pos)
            pred_flat.extend(combo)
            pos += len(combo)
    
    if not true_boundaries:
        return 1.0 if not pred_boundaries else 0.0
    
    # Calculate boundary precision and recall
    correct_boundaries = len(true_boundaries & pred_boundaries)
    precision = correct_boundaries / len(pred_boundaries) if pred_boundaries else 0.0
    recall = correct_boundaries / len(true_boundaries) if true_boundaries else 0.0
    
    # F1 score for boundaries
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def combo_sequence_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate sequence accuracy across all combos (positional accuracy of all strikes)."""
    # Flatten both
    true_flat = []
    for combo in true_combos:
        true_flat.extend(combo)
    
    pred_flat = []
    for combo in predicted_combos:
        pred_flat.extend(combo)
    
    if not pred_flat or not true_flat:
        return 0.0
    
    # Calculate positional accuracy
    min_len = min(len(pred_flat), len(true_flat))
    if min_len == 0:
        return 0.0
    
    matches = sum(1 for i in range(min_len) if pred_flat[i] == true_flat[i])
    return matches / len(true_flat)

def combo_wise_accuracy(true_combos: List[List[str]], predicted_combos: List[List[str]]) -> float:
    """Calculate accuracy of individual combos (how many combos match exactly)."""
    if not true_combos:
        return 1.0 if not predicted_combos else 0.0
    if not predicted_combos:
        return 0.0
    
    # Match combos (order matters)
    min_len = min(len(true_combos), len(predicted_combos))
    exact_matches = sum(1 for i in range(min_len) if true_combos[i] == predicted_combos[i])
    
    return exact_matches / len(true_combos)

print("Evaluation metrics defined!")


Evaluation metrics defined!


In [6]:
# Evaluate on validation set
print(f"Evaluating on {len(val_examples)} validation examples...")
print(f"Using {REQUEST_DELAY}s delay between videos to avoid rate limits")
print(f"(Agent makes multiple API calls per video, so longer delays are needed)\n")

results = []
exact_matches = 0
total_accuracy = 0.0

for i, example in enumerate(val_examples):
    try:
        # Add rate limiting delay (except for first request)
        if i > 0:
            print(f"\nWaiting {REQUEST_DELAY}s to avoid rate limits...")
            time.sleep(REQUEST_DELAY)
        
        print(f"\n{'='*60}")
        print(f"Processing {i+1}/{len(val_examples)}: {Path(example['video_path']).name}")
        print(f"{'='*60}")
        
        # Classify video using agent
        predicted_combos = classify_video_agent(example['video_path'])
        
        true_combos = example['combos']
        exact_match = exact_match_metric(true_combos, predicted_combos)
        combo_count_acc = combo_count_accuracy(true_combos, predicted_combos)
        boundary_acc = combo_boundary_accuracy(true_combos, predicted_combos)
        sequence_acc = combo_sequence_accuracy(true_combos, predicted_combos)
        combo_wise_acc = combo_wise_accuracy(true_combos, predicted_combos)
        
        if exact_match:
            exact_matches += 1
        total_accuracy += sequence_acc
        
        results.append({
            'video_path': example['video_path'],
            'true_combos': true_combos,
            'predicted_combos': predicted_combos,
            'exact_match': exact_match,
            'combo_count_accuracy': combo_count_acc,
            'combo_boundary_accuracy': boundary_acc,
            'sequence_accuracy': sequence_acc,
            'combo_wise_accuracy': combo_wise_acc
        })
        
        print(f"True:  {true_combos}")
        print(f"Pred:  {predicted_combos}")
        print(f"  Exact Match: {'✓' if exact_match else '✗'}")
        print(f"  Sequence Acc: {sequence_acc:.2%}")
        print(f"  Combo Count Acc: {combo_count_acc:.2%}")
        print(f"  Combo Boundary Acc: {boundary_acc:.2%}")
        print(f"  Combo-wise Acc: {combo_wise_acc:.2%}")
            
    except Exception as e:
        print(f"Error processing {example['video_path']}: {e}\n")
        results.append({
            'video_path': example['video_path'],
            'true_combos': example['combos'],
            'predicted_combos': [],
            'exact_match': False,
            'combo_count_accuracy': 0.0,
            'combo_boundary_accuracy': 0.0,
            'sequence_accuracy': 0.0,
            'combo_wise_accuracy': 0.0,
            'error': str(e)
        })

print("\n" + "="*60)
print("Evaluation complete!")
print("="*60)


Evaluating on 24 validation examples...
Using 60s delay between videos to avoid rate limits
(Agent makes multiple API calls per video, so longer delays are needed)


Processing 1/24: 1.mov
[DEBUG] analyze_video returned: [["jab"], ["jab"], ["cross"], ["jab"], ["cross"]]...
True:  [['jab'], ['jab', 'cross', 'lead_hook'], ['jab', 'cross', 'lead_hook', 'lead_hook'], ['jab', 'lead_uppercut', 'cross', 'lead_hook']]
Pred:  [['jab'], ['jab'], ['cross'], ['jab'], ['cross']]
  Exact Match: ✗
  Sequence Acc: 25.00%
  Combo Count Acc: 80.00%
  Combo Boundary Acc: 66.67%
  Combo-wise Acc: 25.00%

Waiting 60s to avoid rate limits...

Processing 2/24: 2.mov
[DEBUG] analyze_video returned: [["jab", "cross", "roundhouse"], ["jab", "cross", "teep"]]...
True:  [['cross', 'lead_hook', 'cross'], ['jab', 'cross', 'lead_uppercut', 'roundhouse'], ['jab', 'lead_elbow', 'roundhouse']]
Pred:  [['jab', 'cross', 'roundhouse'], ['jab', 'cross', 'teep']]
  Exact Match: ✗
  Sequence Acc: 20.00%
  Combo Count Acc: 66

In [7]:
# Calculate and display metrics
exact_match_accuracy = exact_matches / len(val_examples) if val_examples else 0
average_sequence_accuracy = total_accuracy / len(val_examples) if val_examples else 0

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"Exact Match Accuracy: {exact_match_accuracy:.2%} ({exact_matches}/{len(val_examples)})")
print(f"Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print()

# Display some example predictions
print("Sample Predictions:")
print("-" * 60)
for i, result in enumerate(results[:5], 1):
    print(f"\nExample {i}:")
    print(f"  Video: {Path(result['video_path']).name}")
    print(f"  True:  {result['true_combos']}")
    print(f"  Pred:  {result['predicted_combos']}")
    print(f"  Exact Match: {'✓' if result['exact_match'] else '✗'}")
    print(f"  Sequence Acc: {result.get('sequence_accuracy', 0.0):.2%}")


EVALUATION RESULTS
Exact Match Accuracy: 0.00% (0/24)
Average Sequence Accuracy: 12.23%

Sample Predictions:
------------------------------------------------------------

Example 1:
  Video: 1.mov
  True:  [['jab'], ['jab', 'cross', 'lead_hook'], ['jab', 'cross', 'lead_hook', 'lead_hook'], ['jab', 'lead_uppercut', 'cross', 'lead_hook']]
  Pred:  [['jab'], ['jab'], ['cross'], ['jab'], ['cross']]
  Exact Match: ✗
  Sequence Acc: 25.00%

Example 2:
  Video: 2.mov
  True:  [['cross', 'lead_hook', 'cross'], ['jab', 'cross', 'lead_uppercut', 'roundhouse'], ['jab', 'lead_elbow', 'roundhouse']]
  Pred:  [['jab', 'cross', 'roundhouse'], ['jab', 'cross', 'teep']]
  Exact Match: ✗
  Sequence Acc: 20.00%

Example 3:
  Video: 4.mov
  True:  [['teep', 'roundhouse'], ['lead_uppercut', 'cross', 'lead_hook', 'roundhouse'], ['jab', 'cross', 'teep', 'elbow']]
  Pred:  [['teep'], ['roundhouse'], ['jab', 'cross', 'roundhouse']]
  Exact Match: ✗
  Sequence Acc: 30.00%

Example 4:
  Video: 5.mov
  True:  [['

In [8]:
# Calculate per-strike accuracy (flatten all combos)
strike_stats = defaultdict(lambda: {'correct': 0, 'total': 0, 'predicted': 0})

for result in results:
    true_combos = result['true_combos']
    pred_combos = result['predicted_combos']
    
    # Flatten true combos
    true_flat = []
    for combo in true_combos:
        true_flat.extend(combo)
    
    # Flatten predicted combos
    pred_flat = []
    for combo in pred_combos:
        pred_flat.extend(combo)
    
    # Count true strikes
    for strike in true_flat:
        strike_stats[strike]['total'] += 1
    
    # Count predicted strikes
    for strike in pred_flat:
        strike_stats[strike]['predicted'] += 1
    
    # Count correct predictions at each position
    min_len = min(len(true_flat), len(pred_flat))
    for i in range(min_len):
        if true_flat[i] == pred_flat[i]:
            strike_stats[true_flat[i]]['correct'] += 1

print("\nPer-Strike Accuracy:")
print("-" * 60)
for strike in sorted(strike_stats.keys()):
    stats = strike_stats[strike]
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {strike:20s}: {accuracy:6.2%} ({stats['correct']}/{stats['total']} correct, {stats['predicted']} predicted)")



Per-Strike Accuracy:
------------------------------------------------------------
  cross               : 15.38% (8/52 correct, 27 predicted)
  elbow               :  0.00% (0/1 correct, 0 predicted)
  jab                 : 24.19% (15/62 correct, 32 predicted)
  knee                :  0.00% (0/2 correct, 2 predicted)
  lead_elbow          :  0.00% (0/13 correct, 0 predicted)
  lead_hook           :  8.82% (3/34 correct, 5 predicted)
  lead_roundhouse     :  0.00% (0/2 correct, 0 predicted)
  lead_uppercut       :  0.00% (0/14 correct, 0 predicted)
  low_kick            :  0.00% (0/3 correct, 0 predicted)
  rear_elbow          :  0.00% (0/22 correct, 0 predicted)
  rear_hook           :  0.00% (0/2 correct, 2 predicted)
  rear_uppercut       :  0.00% (0/1 correct, 0 predicted)
  roundhouse          : 10.53% (2/19 correct, 13 predicted)
  spinning_elbow      :  0.00% (0/3 correct, 0 predicted)
  switch_knee         :  0.00% (0/3 correct, 0 predicted)
  teep                : 18.18% (2/11

## Section 5: Results and Comparison


In [9]:
# Save evaluation results (same format as DSPy and baseline notebooks)
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Calculate average metrics
avg_combo_count_acc = sum(r.get('combo_count_accuracy', 0.0) for r in results) / len(results) if results else 0.0
avg_boundary_acc = sum(r.get('combo_boundary_accuracy', 0.0) for r in results) / len(results) if results else 0.0
avg_combo_wise_acc = sum(r.get('combo_wise_accuracy', 0.0) for r in results) / len(results) if results else 0.0

# Save evaluation results
results_data = {
    'metrics': {
        'exact_match_accuracy': exact_match_accuracy,
        'average_sequence_accuracy': average_sequence_accuracy,
        'average_combo_count_accuracy': avg_combo_count_acc,
        'average_combo_boundary_accuracy': avg_boundary_acc,
        'average_combo_wise_accuracy': avg_combo_wise_acc,
        'total_examples': len(val_examples),
        'exact_matches': exact_matches
    },
    'results': results,
    'strike_stats': {k: dict(v) for k, v in strike_stats.items()}
}

with open(output_dir / "multiple_agent_evaluation_results.json", 'w') as f:
    json.dump(results_data, f, indent=2)

print(f"Results saved to {output_dir / 'multiple_agent_evaluation_results.json'}")


Results saved to results\multiple_agent_evaluation_results.json


In [10]:
# Display summary
print("\n" + "=" * 60)
print("AGENT EVALUATION SUMMARY")
print("=" * 60)
print(f"Model: Gemini 2.5 Flash (LangGraph Agent)")
print(f"Approach: Agent-based with tools (analyze_video, extract_pose_data)")
print(f"Training examples: {len(train_examples)}")
print(f"Validation examples: {len(val_examples)}")
print(f"\nFinal Performance:")
print(f"  Exact Match Accuracy: {exact_match_accuracy:.2%}")
print(f"  Average Sequence Accuracy: {average_sequence_accuracy:.2%}")
print(f"  Average Combo Count Accuracy: {avg_combo_count_acc:.2%}")
print(f"  Average Combo Boundary Accuracy: {avg_boundary_acc:.2%}")
print(f"  Average Combo-wise Accuracy: {avg_combo_wise_acc:.2%}")
print(f"\nResults saved to: {output_dir}/multiple_agent_evaluation_results.json")
print("\nNote: This evaluation tests the agent's ability to recognize combo boundaries.")
print("Compare these results with DSPy and baseline classifier results for evaluation.")
print("=" * 60)



AGENT EVALUATION SUMMARY
Model: Gemini 2.5 Flash (LangGraph Agent)
Approach: Agent-based with tools (analyze_video, extract_pose_data)
Training examples: 7
Validation examples: 24

Final Performance:
  Exact Match Accuracy: 0.00%
  Average Sequence Accuracy: 12.23%
  Average Combo Count Accuracy: 66.92%
  Average Combo Boundary Accuracy: 48.17%
  Average Combo-wise Accuracy: 3.26%

Results saved to: results/multiple_agent_evaluation_results.json

Note: This evaluation tests the agent's ability to recognize combo boundaries.
Compare these results with DSPy and baseline classifier results for evaluation.
